# Project 5 — The Out-of-Sample Test (the destruction test)

This is the most important validation step in the whole roadmap.

In Project 4, momentum looked like it worked. But you measured it on the **same data you
found it in** — so of course it fit. That's drawing the target around where the arrow already
landed. This notebook does the honest thing:

1. **Split history in two.** An early **training** period and a later **test** period.
2. **Decide the strategy using ONLY the training set** (which momentum settings look good).
3. **Run that exact strategy, untouched, on the test set** it has never seen.
4. **Compare.** If the edge holds out-of-sample, respect it. If it collapses, you just caught
   yourself fitting noise — for free, before risking a cent.

> Run top to bottom, Shift+Enter, kernel = **quant**.

## Step 1 — Load data and split history into train / test

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

prices = pd.read_csv("../data/prices.csv", index_col=0, parse_dates=True)

monthly_price = prices.resample("ME").last()
monthly_ret   = monthly_price.pct_change()

# THE SPLIT: train on the earlier years, test on the later years.
# The strategy will be allowed to "see" only train when we make decisions.
SPLIT_DATE = "2021-01-01"          # everything before = train, after = test

train_price = monthly_price[monthly_price.index <  SPLIT_DATE]
test_price  = monthly_price[monthly_price.index >= SPLIT_DATE]

print(f"Train: {train_price.index.min().date()} -> {train_price.index.max().date()}  ({len(train_price)} months)")
print(f"Test:  {test_price.index.min().date()} -> {test_price.index.max().date()}  ({len(test_price)} months)")

## Step 2 — A reusable momentum backtester

We wrap the Project 4 logic into one function so we can run the *identical* rule on any slice
of data. Note the look-ahead guard (`.shift(1)`) is still here — it never leaves.

In [ ]:
def momentum_strategy(mprice, lookback, top_k):
    """Equal-weight 'winners' portfolio, rebalanced monthly. Returns the monthly return series."""
    mret = mprice.pct_change()
    signal = mprice / mprice.shift(lookback) - 1          # trailing-return momentum
    ranks = signal.rank(axis=1, ascending=False)          # 1 = strongest
    winners = (ranks <= top_k)
    held = winners.shift(1)                               # hold next month (no look-ahead)
    held = held.astype(float)
    port = (mret * held).sum(axis=1) / held.sum(axis=1)
    return port.replace([np.inf, -np.inf], np.nan).dropna()

def sharpe(ret):
    ret = ret.dropna()
    if ret.std() == 0 or len(ret) == 0:
        return np.nan
    return (ret.mean() * 12) / (ret.std() * np.sqrt(12))

# benchmark for each slice: hold all 10 equally
def benchmark(mprice):
    return mprice.pct_change().mean(axis=1).dropna()

print("Functions ready.")

## Step 3 — TRAIN: pick the 'best' settings using only the training data

This is the part where it's legitimate to search — but *only* on the training set. We try a
grid of lookback / top_k combos and find the one with the highest Sharpe **in-sample**.

This is also exactly where overfitting is born: try enough combos and one will look great by
luck. That's fine *here*, because the test set (which we haven't touched) is about to judge it.

In [ ]:
lookbacks = [3, 6, 9, 12]
top_ks    = [2, 3, 4]

results = []
for lb in lookbacks:
    for k in top_ks:
        s = sharpe(momentum_strategy(train_price, lb, k))
        results.append({"lookback": lb, "top_k": k, "train_sharpe": s})

grid = pd.DataFrame(results).sort_values("train_sharpe", ascending=False).reset_index(drop=True)
print("Best settings found ON THE TRAINING DATA:")
best = grid.iloc[0]
print(f"  lookback = {int(best.lookback)} months, top_k = {int(best.top_k)},  train Sharpe = {best.train_sharpe:.2f}")
grid

## Step 4 — TEST: run those exact settings on data the strategy has never seen

We lock in the winning lookback/top_k from training and apply them, unchanged, to the test
period. **This is the moment of truth.** We need a little history before the test window so the
momentum signal exists on day one of the test, so we feed the full series but only score the
test slice.

In [ ]:
BEST_LB = int(best.lookback)
BEST_K  = int(best.top_k)

# run on the FULL series, then slice out only the test period for scoring
full_strat = momentum_strategy(monthly_price, BEST_LB, BEST_K)
full_bench = benchmark(monthly_price)

test_strat = full_strat[full_strat.index >= SPLIT_DATE]
test_bench = full_bench[full_bench.index >= SPLIT_DATE]

train_strat = full_strat[full_strat.index < SPLIT_DATE]

summary = pd.DataFrame({
    "TRAIN  strategy":  [sharpe(train_strat)],
    "TEST   strategy":  [sharpe(test_strat)],
    "TEST   benchmark": [sharpe(test_bench)],
}, index=["Sharpe"]).T.round(2)

print(f"Settings locked from training: lookback={BEST_LB}, top_k={BEST_K}\n")
summary

## Step 5 — Read the verdict

Three numbers to compare:

- **TRAIN strategy Sharpe** — how good it looked when we were fitting. Always flattering.
- **TEST strategy Sharpe** — how it did on unseen data. **This is the only one that counts.**
- **TEST benchmark Sharpe** — just holding all 10. The bar to beat.

The honest outcomes:
- **Test ≈ Train, and Test beats benchmark** -> the edge might be real. (Rare and exciting — and still needs more breadth before you'd trust it.)
- **Test << Train** -> the training result was largely overfitting. The drop *is* the lesson.
- **Test < benchmark** -> the strategy doesn't beat doing nothing on fresh data. Bin it.

In [ ]:
eq_strat = (1 + test_strat).cumprod()
eq_bench = (1 + test_bench).cumprod()

plt.figure(figsize=(12, 6))
plt.plot(eq_strat.index, eq_strat, label=f"Strategy (lb={BEST_LB}, k={BEST_K})", lw=1.6)
plt.plot(eq_bench.index, eq_bench, label="Benchmark (hold all 10)", lw=1.6)
plt.title("OUT-OF-SAMPLE: $1 over the test period the strategy never saw")
plt.ylabel("Value ($, started at 1)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

---
## Why this is the whole game

You've now built the full quant loop end to end:
- **P1** clean data -> **P2** honest metrics -> **P3** a strategy that lost to doing nothing
  -> **P4** a strategy that won, maybe for fake reasons -> **P5** the test that tells the two apart.

The gap between **train Sharpe** and **test Sharpe** has a name: it's the cost of overfitting,
made visible. Professional quants obsess over shrinking that gap, because a strategy that only
works in-sample is worse than useless — it's a confident way to lose money.

**The one habit to carry forever:** never trust a result measured on the data you found it in.
Split, hide, test. If it survives unseen data *and* more breadth, then — and only then — start
to believe it.

### Experiment
- Move `SPLIT_DATE` earlier (`2020-01-01`) or later (`2022-06-01`). Does the verdict change?
  If it swings around with the split, that itself tells you the 'edge' is fragile.
- Notice how big the train-vs-test Sharpe gap is. That gap is your overfitting, quantified.

### Where you'd go next (beyond this arc)
The natural follow-ups, when you're ready: **more breadth** (pull 50–100+ ASX stocks so 3-stock
luck can't dominate), **walk-forward testing** (re-fit periodically as you roll through time),
and **Monte Carlo** (bootstrap the returns to see the *range* of outcomes, not just one path).